# Retrieval

This notebook is made to verify the RAG retriever's model for any combination of model and index.
Change the three config variables to switch between encoders and index variants with no other edits.

**Two retrieval behaviours:**
- **Test-time** (`chunk_id=None`): sample is not in the index → return the k closest chunks directly.
- **Train-time** (`chunk_id=int`): sample IS in the index → search k+1 samples, remove the self-match by id, return k closest chunks.

## 1. Imports

Some librairies migth not being used as we modify the notebook, remove them if you like

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from tqdm import tqdm

## 2. Configuration

In this notebook, we propose to test **one** vector-DB/retriever, choosable by changing the ``MODEL_FAMILY`` (model used to encode the vector DB), ``FINE_TUNE`` (which model ? the base one or one already fine-tuned to classify on a specific dataset) and ``INDEX_SPLIT`` (training chunks, documents chunks or both) variables.  

You can also change `K` (number of samples to retreive) and `THRESHOLD` (cosine-similarity's threshold to retreive).



In [5]:
MODEL_FAMILY = "roberta"   # "bert" | "roberta" | "hatebert"
FINE_TUNE    = "base"    # "base" | "IHC" | "ISHate" | "Vicomtech"
INDEX_SPLIT  = "training"  # "training" | "documents" | "full"

K         = 3
THRESHOLD = 0.95

In [6]:
# Do not edit
HF_IDS = {
    "bert":     "bert-base-uncased",
    "roberta":  "roberta-base",
    "hatebert": "GroNLP/hateBERT",
}
WEIGHTS_PREFIXES = {
    "bert":     "bert-base-uncased",
    "roberta":  "roberta-base",
    "hatebert": "hateBERT",
}

hf_id       = HF_IDS[MODEL_FAMILY]
index_path  = f"index/{MODEL_FAMILY}/{FINE_TUNE}/vdb_{INDEX_SPLIT}.faiss"
lookup_path = f"index/lookup_{INDEX_SPLIT}.json"
weights_path = (
    f"../weights/{WEIGHTS_PREFIXES[MODEL_FAMILY]}_{FINE_TUNE}"
    if FINE_TUNE != "base" else None
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {device}")
print(f"Model family : {MODEL_FAMILY}  ({hf_id})")
print(f"Fine-tune    : {FINE_TUNE}")
print(f"Index split  : {INDEX_SPLIT}")
print(f"Index path   : {index_path}")
print(f"Weights path : {weights_path or '(HuggingFace base weights)'}")

Device       : cpu
Model family : roberta  (roberta-base)
Fine-tune    : base
Index split  : training
Index path   : index/roberta/base/vdb_training.faiss
Weights path : (HuggingFace base weights)


## 3. Load Model, Index & Documents

If `FINE_TUNE`parameter is "base", we need to load the model from HuggingFace otherwise we load it from the folder weigths.  
We load also the index from the `ìndex/` folder and the lookup table to do the masking at training-time

In [9]:
EXPECTED_MODEL_TYPES = {
    "bert":     "bert",
    "roberta":  "roberta",
    "hatebert": "bert",  # hateBERT is BERT-based architecture
}

tokenizer = AutoTokenizer.from_pretrained(hf_id)

if FINE_TUNE == "base":
    model = AutoModel.from_pretrained(hf_id)
else:
    weight_files = ["model.safetensors", "pytorch_model.bin"]
    found = [f for f in weight_files if os.path.isfile(os.path.join(weights_path, f))]
    if not found:
        raise FileNotFoundError(
            f"No model weights found in '{weights_path}'. "
            f"Expected one of {weight_files}. "
            f"Only tokenizer files are present — fine-tuned weights were never saved there."
        )
    full_model = AutoModelForSequenceClassification.from_pretrained(weights_path)
    model = full_model.base_model
    del full_model
model = model.eval().to(device)

index = faiss.read_index(index_path)
print(f"Index loaded  — {index.ntotal:,} vectors  dim={index.d}")

with open(lookup_path) as f:
    documents = json.load(f)
print(f"Lookup loaded — {len(documents):,} entries")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Index loaded  — 67,864 vectors  dim=768
Lookup loaded — 67,864 entries


## 4. Retrieval Functions



In [ ]:
# The functions are now available in the `rag.py` file
from rag import encode, retrieve_top_k, retrieve_by_threshold, retrieve_top_k_above_threshold

## 5. Validation — Test-time Behaviour

Queries without a `chunk_id` (simulating the test set, where tweets are not in the index).
Expected:
- Hate tweets → hateful, semantically relevant neighbors.
- "i dont like tomatoes" → weak, unrelated neighbors (low semantic match).
- "i dont like neggers" → hateful neighbors.

In [ ]:
test_queries = [
    ("HATE",     "jews are now in full control : canadian man goes to jail for posting on white nationalist websites"),
    ("HATE",     "i dont like neggers"),
    ("NOT HATE", "i dont like tomatoes"),
    ("NOT HATE", "Charges in deadly wreck that killed 5 motorcyclists . Crash takes injured woman s fiance , mom ."),
]

sep = "=" * 70

for label, tweet in test_queries:
    print(f"\n{sep}")
    print(f"[{label}] {tweet}")

    print(f"  -- retrieve_top_k (k={K}) --")
    for text, score in retrieve_top_k(tweet, model, tokenizer, index, documents):
        print(f"    {score:.4f}  {text[:90]}")

    # Uncomment to test retrieve_by_threshold (can return hundreds of results):
    # print(f"  -- retrieve_by_threshold (threshold={THRESHOLD}) --")
    # hits = retrieve_by_threshold(tweet, THRESHOLD, model, tokenizer, index, documents)
    # for text, score in hits or []:
    #     print(f"    {score:.4f}  {text[:90]}")
    # if not hits:
    #     print("    (no results above threshold)")

    print(f"  -- retrieve_top_k_above_threshold (threshold={THRESHOLD}, k=10) --")
    hits = retrieve_top_k_above_threshold(tweet, THRESHOLD, model, tokenizer, index, documents, k=10)
    if hits:
        for text, score in hits:
            print(f"    {score:.4f}  {text[:90]}")
    else:
        print("    (no results above threshold)")

## 6. Validation — Train-time Self-exclusion

Only meaningful when `INDEX_SPLIT == "training"` (tweets are in the index).

For 3 randomly sampled training tweets, verify that:
1. The tweet's own text does **not** appear in the retrieved neighbors when `chunk_id` is passed.
2. It **does** appear (as the top result) when `chunk_id=None` is passed — confirming it was in the index.

In [ ]:
if INDEX_SPLIT != "training":
    print(f"Skipping self-exclusion check — INDEX_SPLIT='{INDEX_SPLIT}' (tweets not in this index).")
else:
    df = pd.read_csv("chunks/chunks_training.csv")

    def strip_prefix(text):
        return text.replace("[hate] ", "", 1).replace("[not hate] ", "", 1)

    sample = df.sample(3, random_state=42)

    for _, row in sample.iterrows():
        raw = strip_prefix(row.text)
        cid = int(row.chunk_id)

        without_exclusion = retrieve_top_k(raw, model, tokenizer, index, documents, chunk_id=None, k=K + 1)
        with_exclusion    = retrieve_top_k(raw, model, tokenizer, index, documents, chunk_id=cid,  k=K)

        self_in_without = any(text == row.text for text, _ in without_exclusion)
        self_in_with    = any(text == row.text for text, _ in with_exclusion)

        print("\n" + "=" * 70)
        print(f"chunk_id={cid}  |  {raw[:80]}")
        print(f"  Self present WITHOUT exclusion : {self_in_without}  (expected True)")
        print(f"  Self present WITH    exclusion : {self_in_with}   (expected False)")
        assert self_in_without, f"chunk_id={cid} not found in index — check index"
        assert not self_in_with, f"Self-exclusion failed for chunk_id={cid}"

    print("\nAll self-exclusion checks passed.")

## 7. Vector-Space Analysis

Metrics that characterise the geometry of the embedding space created by this model/index pair.
All metrics are computed on a random sample of `N_SAMPLE` vectors drawn from the index.

| Metric | Interpretation |
|---|---|
| **Effective rank** | How many dimensions carry meaningful variance (higher = richer space) |
| **Mean pairwise cosine similarity** | Isotropy: lower means vectors are well spread across the sphere |
| **Cosine similarity distribution** | Shape of the similarity landscape |
| **Norm statistics** (before L2-norm) | Whether the model collapses embeddings to similar magnitudes |
| **Intra- vs inter-class similarity** | Whether hate/not-hate form geometrically separated clusters |

In [ ]:
N_SAMPLE = 2000  # number of random vectors to draw from the index

# ── Sample raw (un-normalised) vectors from the index ────────────────────────
# IndexIDMap wraps IndexFlatIP; reconstruct gives L2-normalised vectors since
# we normalised before adding. We re-encode a random sample from the lookup to
# get the raw (pre-norm) norms as well.
rng = np.random.default_rng(0)
all_ids = list(documents.keys())
sample_ids = rng.choice(all_ids, size=min(N_SAMPLE, len(all_ids)), replace=False)
sample_texts = [documents[sid] for sid in sample_ids]

print(f"Encoding {len(sample_texts)} random samples...")
raw_vecs = encode(sample_texts, model, tokenizer)   # shape (N, D), un-normalised

# ── 1. Norm statistics ────────────────────────────────────────────────────────
norms = np.linalg.norm(raw_vecs, axis=1)
print(f"\n── Norm statistics (pre-normalisation) ──")
print(f"  mean : {norms.mean():.4f}")
print(f"  std  : {norms.std():.4f}")
print(f"  min  : {norms.min():.4f}   max : {norms.max():.4f}")

# ── 2. Effective rank ─────────────────────────────────────────────────────────
# Effective rank = exp(entropy of normalised singular value distribution).
# Ranges from 1 (rank-1 collapse) to D (full use of all dimensions).
centered = raw_vecs - raw_vecs.mean(axis=0)
_, sv, _ = np.linalg.svd(centered, full_matrices=False)
sv_norm = sv / sv.sum()
entropy = -np.sum(sv_norm * np.log(sv_norm + 1e-12))
eff_rank = np.exp(entropy)
print(f"\n── Effective rank ──")
print(f"  Effective rank : {eff_rank:.1f}  /  {raw_vecs.shape[1]}  dimensions")

# ── 3. Mean pairwise cosine similarity (isotropy) ─────────────────────────────
normed = raw_vecs / (norms[:, None] + 1e-12)
# Compute all pairwise cosines via matrix multiply; exclude self-pairs
cos_matrix = normed @ normed.T
np.fill_diagonal(cos_matrix, np.nan)
pairwise = cos_matrix[~np.isnan(cos_matrix)]
mean_cos = np.nanmean(pairwise)
std_cos  = np.nanstd(pairwise)
print(f"\n── Pairwise cosine similarity ──")
print(f"  mean : {mean_cos:.4f}   std : {std_cos:.4f}")
print(f"  → {'High anisotropy — vectors cluster tightly' if mean_cos > 0.95 else 'Good spread across the sphere' if mean_cos < 0.85 else 'Moderate spread'}")

# ── 4. Cosine similarity distribution (histogram) ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"Vector space — {MODEL_FAMILY} / {FINE_TUNE} / {INDEX_SPLIT}")

# Subsample pairs to keep the histogram manageable
pair_sample = rng.choice(pairwise, size=min(100_000, len(pairwise)), replace=False)
axes[0].hist(pair_sample, bins=80, color="steelblue", edgecolor="none")
axes[0].axvline(mean_cos, color="red", linestyle="--", label=f"mean={mean_cos:.3f}")
axes[0].set_xlabel("Cosine similarity")
axes[0].set_ylabel("Count")
axes[0].set_title("Pairwise cosine similarity distribution")
axes[0].legend()

axes[1].hist(norms, bins=60, color="darkorange", edgecolor="none")
axes[1].axvline(norms.mean(), color="red", linestyle="--", label=f"mean={norms.mean():.2f}")
axes[1].set_xlabel("L2 norm")
axes[1].set_ylabel("Count")
axes[1].set_title("Embedding norm distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

# ── 5. Intra- vs inter-class similarity ──────────────────────────────────────
hate_mask    = np.array([documents[sid].startswith("[hate]")     for sid in sample_ids])
nothate_mask = np.array([documents[sid].startswith("[not hate]") for sid in sample_ids])

def mean_block_similarity(mask_a, mask_b, cos_mat):
    block = cos_mat[np.ix_(mask_a, mask_b)]
    if mask_a is mask_b:
        np.fill_diagonal(block, np.nan)
    return np.nanmean(block)

if hate_mask.sum() > 1 and nothate_mask.sum() > 1:
    intra_hate    = mean_block_similarity(hate_mask, hate_mask, cos_matrix)
    intra_nothate = mean_block_similarity(nothate_mask, nothate_mask, cos_matrix)
    inter         = mean_block_similarity(hate_mask, nothate_mask, cos_matrix)
    print(f"\n── Intra- vs inter-class cosine similarity ──")
    print(f"  hate    ↔ hate    (intra) : {intra_hate:.4f}")
    print(f"  nothate ↔ nothate (intra) : {intra_nothate:.4f}")
    print(f"  hate    ↔ nothate (inter) : {inter:.4f}")
    sep_ratio = ((intra_hate + intra_nothate) / 2 - inter)
    print(f"  Separation gap (intra_mean − inter) : {sep_ratio:.4f}  {'← good separation' if sep_ratio > 0.01 else '← classes overlap'}")
else:
    print("\n(Not enough labelled samples to compute intra/inter-class similarity)")